# 🚀 Warmup Cache — Pre-generate Analysis Embeddings

Notebook này chạy **local** để pre-cache tất cả analysis embeddings trước khi training.

**Flow:**
1. Load `.env` → Set API key
2. Chạy pipeline tuần tự cho tất cả windows → cache vào `stock_analysis/st_data/`
3. Training sẽ hit cache, không cần gọi LLM API nữa

**Có thể interrupt bất cứ lúc nào** — đã cache sẽ không gọi lại.

## 1. Setup

In [1]:
# Load API key từ .env hoặc nhập thủ công
import logging
import os
from pathlib import Path
from dotenv import load_dotenv

# Load từ stock_analysis/.env nếu có
env_path = Path("stock_analysis/.env")
if env_path.exists():
    load_dotenv(env_path)
    logging.info(f"✓ Loaded .env from {env_path}")
else:
    # Nhập thủ công nếu chưa có .env
    from getpass import getpass
    api_key = getpass("Nhập API_KEY: ")
    env_path.write_text(f"API_KEY={api_key}\n")
    os.environ["API_KEY"] = api_key
    logging.info(f"✓ API_KEY set manually, written to {env_path}")

assert os.environ.get("API_KEY"), "API_KEY chưa được set!"
logging.info("✓ API_KEY ready")

## 2. Load Config & Kiểm tra Data

In [2]:
import time
import logging
import yaml
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

from src.features.preprocessor import load_csv
from stock_analysis import pipeline

logging.basicConfig(level=logging.WARNING, format='%(asctime)s %(levelname)s %(message)s', datefmt='%H:%M:%S')

# Load config
with open('configs/config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

WINDOW = cfg['env']['window']  # 20
MODEL = cfg['analysis']['model']
PATHS = cfg['data']['paths']

print(f'Window: {WINDOW}')
print(f'Model:  {MODEL}')
print(f'Stocks: {[Path(p).stem for p in PATHS]}')

# Kiểm tra data files
for p in PATHS:
    if Path(p).exists():
        df = load_csv(p)
        print(f"  ✓ {Path(p).stem}: {len(df)} rows | {df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()} | Windows: {len(df)-WINDOW}")
    else:
        logging.error(f"  ✗ {p} NOT FOUND!")

Loaded as API: https://473f19b364d6a6a460.gradio.live/
Loaded as API: https://cf4a4cd119eb8adcac.gradio.live/
Window: 20
Model:  unsloth/Qwen2.5-14B-Instruct-bnb-4bit
Stocks: ['VNM', 'FPT', 'VIC', 'HPG']
[Data] VNM.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VNM.csv Close range: 47.97 → 88.46
  ✓ VNM: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266
[Data] FPT.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] FPT.csv Close range: 22.91 → 131.67
  ✓ FPT: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266
[Data] VIC.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VIC.csv Close range: 19.95 → 169.90
  ✓ VIC: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266
[Data] HPG.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] HPG.csv Close range: 9.00 → 32.65
  ✓ HPG: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266


## 3. Chạy Cache (Tuần tự)

Mỗi window gọi pipeline 1 lần. Pipeline tự cache theo hash nội dung report:
- Nếu report đã tồn tại → skip tạo report
- Nếu LLM response đã tồn tại → skip gọi API
- Nếu embedding đã tồn tại → skip tính embedding

→ Chạy lại an toàn, chỉ tốn thời gian cho windows chưa cache.

In [3]:
def warmup_stock(
    stock_id: str,
    csv_path: str,
    window: int = 20,
    skip_every: int = 1,
):
    """
    Cache tất cả windows cho 1 stock, xử lý tuần tự từng window một.
    
    Args:
        skip_every: Cache mỗi N windows (1 = tất cả, 5 = mỗi 5 ngày).
    """
    df = load_csv(csv_path)
    
    success_count = 0
    errors = 0
    error_msgs = []
    
    steps = list(range(window, len(df), skip_every))
    pbar = tqdm(steps, desc=stock_id, unit="win")
    
    for step in pbar:
        start_idx = max(0, step - window + 1)
        date_end = pd.Timestamp(df['date'].iloc[step]).to_pydatetime()
        date_start = pd.Timestamp(df['date'].iloc[start_idx]).to_pydatetime()
        
        try:
            pipeline(
                model=MODEL,
                stock_id=stock_id,
                date_start=date_start,
                date_end=date_end,
            )
            success_count += 1
        except Exception as e:
            logging.error(e)
            errors += 1
            if len(error_msgs) < 5:
                error_msgs.append(f"date={date_end.date()}: {e}")
        
        pbar.set_postfix(ok=success_count, err=errors)
    
    pbar.close()
    
    # Tóm tắt
    print(f"  ✓ {stock_id}: {success_count} ok, {errors} errors (total: {len(steps)})")
    if error_msgs:
        print(f"  Errors (showing first {len(error_msgs)}):")
        for msg in error_msgs:
            print(f"    • {msg}")
    
    return success_count, errors

### Chạy từng stock (có thể interrupt an toàn)

In [4]:
warmup_stock('FPT', 'data/FPT.csv', WINDOW, skip_every=1)

[Data] FPT.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] FPT.csv Close range: 22.91 → 131.67


FPT:   0%|          | 0/1266 [00:00<?, ?win/s]

For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


  ✓ FPT: 1266 ok, 0 errors (total: 1266)


(1266, 0)

In [8]:
from pathlib import Path
import re

def replace_exact_word_in_json(folder_path, target_word="VNM", replacement="that"):
    path = Path(folder_path)
    
    if not path.is_dir():
        print(f"Thư mục '{folder_path}' không tồn tại.")
        return

    # Sử dụng Regex với \b để định vị chính xác từ "this"
    # re.IGNORECASE (tùy chọn): nếu bạn muốn đổi cả "This", "THIS" thành "that"
    pattern = re.compile(rf'\b{re.escape(target_word)}\b') 

    for json_file in path.rglob("*.md"):
        try:
            content = json_file.read_text(encoding="utf-8")
            
            # Kiểm tra xem có chứa chính xác từ "VNM" độc lập hay không
            if pattern.search(content):
                # Thay thế chính xác từ "VNM" thành "that"
                updated_content = pattern.sub(replacement, content)
                
                json_file.write_text(updated_content, encoding="utf-8")
                print(f" Đã cập nhật thành công: {json_file}")
            else:
                print(f" Bỏ qua (không chứa từ '{target_word}' đứng độc lập): {json_file}")
                
        except Exception as e:
            print(f" Lỗi khi xử lý file {json_file}: {e}")

# --- Chạy thử nghiệm ---
thu_muc_target = r"D:\bnquys\Source\stock-trend-prediction\stock_analysis\st_data\VIC\responses"  # Thay bằng đường dẫn thư mục của bạn
replace_exact_word_in_json(thu_muc_target, target_word="VNM", replacement="VIC")

 Bỏ qua (không chứa từ 'VNM' đứng độc lập): D:\bnquys\Source\stock-trend-prediction\stock_analysis\st_data\VIC\responses\015761667d8da9ac2f67b710df4e5f3442e861d88034e203fef1f9b052271b2f.md
 Đã cập nhật thành công: D:\bnquys\Source\stock-trend-prediction\stock_analysis\st_data\VIC\responses\0346c23d5a3341c7c34f16532cbc7e48447bf4989cf6683153aaf352af8a3823.md
 Bỏ qua (không chứa từ 'VNM' đứng độc lập): D:\bnquys\Source\stock-trend-prediction\stock_analysis\st_data\VIC\responses\03609b84f6a1464020106fe7c1874ffba06ad9e327a92a29f2e26e919c6c7e9b.md
 Bỏ qua (không chứa từ 'VNM' đứng độc lập): D:\bnquys\Source\stock-trend-prediction\stock_analysis\st_data\VIC\responses\037e5c0458857fad71a274c829006cdb23c4f301fa7b9284b70115f1d548fe66.md
 Đã cập nhật thành công: D:\bnquys\Source\stock-trend-prediction\stock_analysis\st_data\VIC\responses\04f93aab3013ca814a7e4f7709806723305e85c888b982e4117187579c28b73f.md
 Bỏ qua (không chứa từ 'VNM' đứng độc lập): D:\bnquys\Source\stock-trend-prediction\stock_ana